In [11]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

from churn.config import ID, TARGET, RANDOM_SEED
from churn.pipeline import construir_pipeline

In [3]:
pdf = pd.read_csv("../data/gold_split.csv")
train = pdf[pdf["conjunto"] == "train"]

X_train = train.drop(columns=[ID, TARGET, "conjunto"])
y_train = train[TARGET]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

assert len(train) == 5634, f"{len(train)} filas de train"

In [ ]:
logreg = construir_pipeline(LogisticRegression(random_state=RANDOM_SEED))
scores = cross_val_score(logreg, X_train, y_train, cv=cv, scoring="average_precision")


c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\si

(np.float64(0.654699117752892), np.float64(0.020626247652513315))

In [5]:
print(scores.mean())
print(scores.std())

0.654699117752892
0.020626247652513315


## RandomForest

In [6]:
espacio_rf = {
    "modelo__n_estimators": [200, 400, 600],
    "modelo__max_depth": [4, 6, 8, 10, 15, None],
    "modelo__min_samples_leaf": [1, 5, 10, 20, 50],
    "modelo__max_features": ["sqrt", 0.3, 0.5],
}

In [ ]:
busqueda_gb = RandomizedSearchCV(
    construir_pipeline(RandomForestClassifier(random_state=RANDOM_SEED)),
    param_distributions=espacio_rf,
    n_iter=40,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
busqueda_gb.fit(X_train, y_train)

In [ ]:
mejor = busqueda_gb.best_index_
print(busqueda_gb.best_score_, busqueda_gb.cv_results_["std_test_score"][mejor])
busqueda_gb.best_params_

0.673141734313574 0.020712930848338293


{'modelo__n_estimators': 600,
 'modelo__min_samples_leaf': 10,
 'modelo__max_features': 0.3,
 'modelo__max_depth': 8}

## GradientBoosting

In [9]:
espacio_gb = {
    "modelo__n_estimators": [100, 200, 400],
    "modelo__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "modelo__max_depth": [2, 3, 4, 5],
    "modelo__min_samples_leaf": [1, 10, 20, 50],
    "modelo__subsample": [0.7, 0.85, 1.0],
}

In [12]:
busqueda_gb = RandomizedSearchCV(
    construir_pipeline(GradientBoostingClassifier(random_state=RANDOM_SEED)),
    param_distributions=espacio_gb,
    n_iter=40,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
busqueda_gb.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocesado',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(handle_unknown='ignore',
                                                                                             sparse_output=False),
                                                                               ['gender',
                                                                                'partner',
                                                                                'dependents',
                                                                                'phone_service',
                                                                                'multiple_lines',
                                                                                'internet_service',
                                                                                'online_security',
                                                                                'online_backup',
                                                                                'device_prot...
                                                                                'automatic_payment'])])),
                                             ('modelo',
                                              GradientBoostingClassifier(random_state=42))]),
                   n_iter=40, n_jobs=-1,
                   param_distributions={'modelo__learning_rate': [0.01, 0.03,
                                                                  0.05, 0.1],
                                        'modelo__max_depth': [2, 3, 4, 5],
                                        'modelo__min_samples_leaf': [1, 10, 20,
                                                                     50],
                                        'modelo__n_estimators': [100, 200, 400],
                                        'modelo__subsample': [0.7, 0.85, 1.0]},
                   random_state=42, scoring='average_precision')

In [13]:
mejor_gb = busqueda_gb.best_index_
print(busqueda_gb.best_score_, busqueda_gb.cv_results_["std_test_score"][mejor_gb])
busqueda_gb.best_params_

0.6729004163540162 0.01526061776727842


{'modelo__subsample': 1.0,
 'modelo__n_estimators': 200,
 'modelo__min_samples_leaf': 50,
 'modelo__max_depth': 2,
 'modelo__learning_rate': 0.03}

In [14]:
espacio_gb_borde = {
    "modelo__n_estimators": [200, 400],
    "modelo__learning_rate": [0.03],
    "modelo__max_depth": [1, 2, 3],
    "modelo__min_samples_leaf": [50, 100, 200],
    "modelo__subsample": [1.0],
}

In [15]:
busqueda_gb_borde = RandomizedSearchCV(
    construir_pipeline(GradientBoostingClassifier(random_state=RANDOM_SEED)),
    param_distributions=espacio_gb_borde,
    n_iter=40,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
busqueda_gb_borde.fit(X_train, y_train)

c:\Users\ramir\Documents\GitHub\Ejercicios Libre\telco-churn\telco-churn-prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 18 is smaller than n_iter=40. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocesado',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(handle_unknown='ignore',
                                                                                             sparse_output=False),
                                                                               ['gender',
                                                                                'partner',
                                                                                'dependents',
                                                                                'phone_service',
                                                                                'multiple_lines',
                                                                                'internet_service',
                                                                                'online_security',
                                                                                'online_backup',
                                                                                'device_prot...
                                                                                'tenure_max',
                                                                                'fiber_no_support',
                                                                                'automatic_payment'])])),
                                             ('modelo',
                                              GradientBoostingClassifier(random_state=42))]),
                   n_iter=40, n_jobs=-1,
                   param_distributions={'modelo__learning_rate': [0.03],
                                        'modelo__max_depth': [1, 2, 3],
                                        'modelo__min_samples_leaf': [50, 100,
                                                                     200],
                                        'modelo__n_estimators': [200, 400],
                                        'modelo__subsample': [1.0]},
                   random_state=42, scoring='average_precision')

In [16]:
mejor_gb_borde = busqueda_gb_borde.best_index_
print(busqueda_gb_borde.best_score_, busqueda_gb_borde.cv_results_["std_test_score"][mejor_gb_borde])
busqueda_gb_borde.best_params_

0.6737633870796685 0.01567425259519647


{'modelo__subsample': 1.0,
 'modelo__n_estimators': 400,
 'modelo__min_samples_leaf': 200,
 'modelo__max_depth': 2,
 'modelo__learning_rate': 0.03}

## XGBoost

In [17]:
espacio_xgb = {
    "modelo__n_estimators": [200, 400, 600],
    "modelo__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "modelo__max_depth": [2, 3, 4, 6],
    "modelo__min_child_weight": [1, 5, 10, 20],
    "modelo__subsample": [0.7, 0.85, 1.0],
    "modelo__colsample_bytree": [0.5, 0.7, 1.0],
    "modelo__reg_lambda": [1, 5, 10],
}

In [18]:
modelo_xgb = XGBClassifier(random_state=RANDOM_SEED, n_jobs=1)

In [ ]:
busqueda_xgb = RandomizedSearchCV(
    construir_pipeline(modelo_xgb),
    param_distributions=espacio_xgb,
    n_iter=60,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
busqueda_xgb.fit(X_train, y_train)